In [0]:
from pyspark.sql import functions as F

catalog = spark.sql(
    "SELECT current_catalog() AS catalog"
).first()["catalog"]

bronze_table = f"{catalog}.f1_bronze.api_responses"

pit_bronze_df = (
    spark.table(bronze_table)
    .filter(F.col("entity_type") == "pit_stops")
)

sample_json = (
    pit_bronze_df
    .select("payload_json")
    .first()["payload_json"]
)

pit_schema = (
    spark.range(1)
    .select(
        F.schema_of_json(F.lit(sample_json)).alias("json_schema")
    )
    .first()["json_schema"]
)

parsed_pits_df = pit_bronze_df.withColumn(
    "json_data",
    F.from_json(
        F.col("payload_json"),
        pit_schema
    )
)

In [0]:
pit_race_rows_df = (
    parsed_pits_df
    .select(
        F.explode(
            "json_data.MRData.RaceTable.Races"
        ).alias("race")
    )
)

pit_rows_df = (
    pit_race_rows_df
    .select(
        F.col("race.season").cast("int").alias("season"),
        F.col("race.round").cast("int").alias("round"),
        F.explode("race.PitStops").alias("pit")
    )
)

In [0]:
silver_pits_df = (
    pit_rows_df
    .select(
        F.concat_ws(
            "_",
            F.col("season").cast("string"),
            F.lpad(F.col("round").cast("string"), 2, "0")
        ).alias("race_id"),

        "season",
        "round",

        F.col("pit.driverId").alias("driver_id"),
        F.col("pit.stop").cast("int").alias("stop_number"),
        F.col("pit.lap").cast("int").alias("pit_lap"),
        F.col("pit.time").alias("pit_time"),
        F.col("pit.duration").alias("duration_text")
    )
    .withColumn(
        "duration_ms",
        F.when(
            F.col("duration_text").contains(":"),
            (
                F.split("duration_text", ":").getItem(0).cast("double") * 60000
                + F.split("duration_text", ":").getItem(1).cast("double") * 1000
            ).cast("long")
        ).otherwise(
            (F.col("duration_text").cast("double") * 1000).cast("long")
        )
    )
)

display(silver_pits_df)

In [0]:
# 1. A driver should not have the same stop number twice in one race.
duplicate_pits_df = (
    silver_pits_df
    .groupBy("race_id", "driver_id", "stop_number")
    .count()
    .filter("count > 1")
)

print("Duplicate pit-stop keys:", duplicate_pits_df.count())
display(duplicate_pits_df)

In [0]:
# 2. Pit-stop lap numbers must be positive.
invalid_pit_laps_df = silver_pits_df.filter(
    F.col("pit_lap").isNull()
    | (F.col("pit_lap") <= 0)
    | (F.col("pit_lap") > 250)
)

print("Invalid pit-stop laps:", invalid_pit_laps_df.count())
display(invalid_pit_laps_df)

In [0]:
# 3. Stop numbers and durations must be valid.
invalid_pit_values_df = silver_pits_df.filter(
    F.col("stop_number").isNull()
    | (F.col("stop_number") <= 0)
    | F.col("duration_ms").isNull()
    | (F.col("duration_ms") <= 0)
)

print("Invalid pit-stop values:", invalid_pit_values_df.count())
display(invalid_pit_values_df)

In [0]:
catalog = spark.sql(
    "SELECT current_catalog() AS catalog"
).first()["catalog"]

silver_pit_table = f"{catalog}.f1_silver.pit_stops"

silver_pits_to_save_df = silver_pits_df.withColumn(
    "loaded_at",
    F.current_timestamp()
)

(
    silver_pits_to_save_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_pit_table)
)

print(f"Saved Silver table: {silver_pit_table}")

In [0]:
saved_pits_df = spark.table(silver_pit_table)

print("Pit-stop rows:", saved_pits_df.count())

display(
    saved_pits_df
    .select(
        "race_id",
        "driver_id",
        "stop_number",
        "pit_lap",
        "duration_text",
        "duration_ms"
    )
    .orderBy("driver_id", "stop_number")
)